# Tutorial Multi-GNN: Detección de Lavado de Dinero con Graph Neural Networks

Este notebook te guiará paso a paso por el código del repositorio **IBM Multi-GNN** para que puedas entender y ejecutar los modelos de detección de lavado de dinero (AML - Anti-Money Laundering).

## ¿Qué aprenderás?

1. **Conceptos básicos**: Qué son las GNNs y por qué son útiles para transacciones financieras
2. **Carga de datos**: Cómo se procesan las transacciones financieras como grafos
3. **Modelos**: 4 arquitecturas diferentes (GINe, GATe, PNA, RGCN)
4. **Entrenamiento**: Cómo se entrenan estos modelos paso a paso
5. **Evaluación**: Métricas y cómo interpretar resultados

## Estructura del notebook

- **Parte 1**: Setup y conceptos básicos
- **Parte 2**: Exploración de datos
- **Parte 3**: Comprensión de los modelos
- **Parte 4**: Entrenamiento completo
- **Parte 5**: Inferencia y predicción

---
# Parte 1: Setup y Conceptos Básicos

## 1.1 ¿Qué son las Graph Neural Networks (GNNs)?

Las **GNNs** son redes neuronales diseñadas para trabajar con datos en forma de grafo:
- **Nodos**: Cuentas bancarias
- **Aristas**: Transacciones entre cuentas
- **Características**: Cantidad, moneda, timestamp, etc.

### ¿Por qué GNNs para detección de lavado de dinero?

El lavado de dinero no es solo una transacción sospechosa aislada, sino **patrones de comportamiento en la red**:
- Múltiples transacciones pequeñas (smurfing)
- Cadenas de transferencias para ofuscar origen
- Patrones temporales anómalos

Las GNNs pueden capturar estos patrones porque:
1. **Message Passing**: Los nodos intercambian información con sus vecinos
2. **Agregación**: Cada nodo aprende del contexto de su vecindario
3. **Edge Prediction**: Predicen si una transacción (arista) es ilícita

## 1.2 Instalación de dependencias

In [ ]:
# Primero, vamos a verificar si tenemos las librerías necesarias
import sys
import subprocess

def check_and_install():
    """
    Verifica e instala las dependencias necesarias.
    """
    print("Verificando dependencias...\n")
    
    # Lista de paquetes a verificar
    packages = [
        'torch',
        'torch_geometric',
        'numpy',
        'pandas',
        'matplotlib',
        'scikit-learn',
        'tqdm'
    ]
    
    missing = []
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
            print(f"✓ {package} instalado")
        except ImportError:
            print(f"✗ {package} NO instalado")
            missing.append(package)
    
    if missing:
        print(f"\n⚠️ Faltan paquetes: {', '.join(missing)}")
        print("\nPara instalarlos, ejecuta en tu terminal:")
        print("conda env create -f Multi-GNN/env.yml")
        print("conda activate multignn")
    else:
        print("\n✓ Todas las dependencias están instaladas!")

check_and_install()

## 1.3 Importar librerías

In [ ]:
# Librerías estándar
import os
import sys
import json
from pathlib import Path

# Añadir el directorio Multi-GNN al path para importar sus módulos
sys.path.insert(0, os.path.join(os.getcwd(), 'Multi-GNN'))

# Ciencia de datos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

# PyTorch Geometric
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

# Scikit-learn para métricas
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# Configurar estilo de visualización
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
# Parte 2: Exploración de Datos

## 2.1 Descargar y preparar los datos de Kaggle

Los datos provienen de Kaggle: **IBM Transactions for Anti-Money Laundering (AML)**

**Link**: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml

### 📥 Pasos para obtener los datos:

#### Opción 1: Descarga manual

1. **Crear cuenta en Kaggle** (si no tienes): https://www.kaggle.com/account/login
2. **Descargar el dataset**:
   - Ve al link del dataset
   - Click en "Download" (descargará un archivo ZIP ~200MB)
3. **Descomprimir**:
   - Extrae los archivos CSV   - Busca el archivo de transacciones (ej: `HI-Small_Trans.csv`, `HI-Medium_Trans.csv`)

#### Opción 2: Descarga con Kaggle API (recomendado)

```bash
# 1. Instalar kaggle API
pip install kaggle

# 2. Configurar credenciales (descarga tu API token de Kaggle)
# Ve a: https://www.kaggle.com/settings -> Create New Token
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json

# 3. Descargar dataset
kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml
unzip ibm-transactions-for-anti-money-laundering-aml.zip -d data/kaggle_raw/
```

### 🔧 Formateo de datos

**Formato original de Kaggle:**
```
Timestamp (str) | From Bank | From Account | To Bank | To Account | 
Amount Received | Receiving Currency | Amount Paid | Payment Currency | 
Payment Format | Is Laundering
```

**Formato procesado** (usado por Multi-GNN):
```
EdgeID | from_id | to_id | Timestamp (unix) | Amount Sent | Sent Currency | 
Amount Received | Received Currency | Payment Format | Is Laundering
```

El script `Multi-GNN/format_kaggle_files.py` hace esta conversión automáticamente.

### 📊 Datasets disponibles en Kaggle:

| Dataset | Nombre archivo | Nodos | Transacciones | % Ilícitas | Tamaño |
|---------|---------------|-------|---------------|------------|--------|
| HI-Small | HI-Small_Trans.csv | ~4K | ~200K | ~2% | ~40MB |
| HI-Medium | HI-Medium_Trans.csv | ~10K | ~500K | ~2% | ~100MB |
| HI-Large | HI-Large_Trans.csv | ~30K | ~1.5M | ~2% | ~300MB |

**Recomendación**: Empieza con **HI-Small** para aprender, luego escala a datasets más grandes.

Para este tutorial, **usaremos datos reales si están disponibles**, o crearemos datos sintéticos para demostración.

In [ ]:
# Configurar rutas
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

KAGGLE_RAW_DIR = DATA_DIR / "kaggle_raw"
KAGGLE_RAW_DIR.mkdir(exist_ok=True)

# Buscar archivos de Kaggle (crudos o formateados)
FORMATTED_FILE = DATA_DIR / "formatted_transactions.csv"

# Buscar archivos crudos de Kaggle
KAGGLE_FILES = list(KAGGLE_RAW_DIR.glob("*Trans.csv"))

print("🔍 BUSCANDO DATOS...")
print("=" * 70)

# Verificar si ya tenemos datos formateados
if FORMATTED_FILE.exists():
    print(f"✓ Datos formateados encontrados: {FORMATTED_FILE}")
    print(f"  Tamaño: {FORMATTED_FILE.stat().st_size / 1024 / 1024:.1f} MB")
    USE_REAL_DATA = True
    
# Verificar si tenemos datos crudos de Kaggle para formatear
elif KAGGLE_FILES:
    print(f"✓ Datos crudos de Kaggle encontrados: {len(KAGGLE_FILES)} archivo(s)")
    for f in KAGGLE_FILES:
        print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")
    
    # Preguntar cuál archivo usar
    print("\n📝 Necesitamos formatear los datos usando el script de Multi-GNN")
    print("\nEjecuta en tu terminal:")
    print(f"  python Multi-GNN/format_kaggle_files.py {KAGGLE_FILES[0]}")
    print(f"\nEsto creará: {KAGGLE_FILES[0].parent}/formatted_transactions.csv")
    print("\n⚠️ Por ahora, usaremos datos sintéticos para demostración...")
    USE_REAL_DATA = False

else:
    print("✗ No se encontraron datos de Kaggle")
    print("\n📝 OPCIONES:") 
    print("\n1️⃣ Descarga manual:")
    print("   - Ve a: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
    print("   - Descarga y descomprime en:", KAGGLE_RAW_DIR)
    print("   - Ejecuta: python Multi-GNN/format_kaggle_files.py data/kaggle_raw/HI-Small_Trans.csv")
    print("   - Los datos formateados se guardarán automáticamente")
    
    print("\n2️⃣ Con Kaggle API:")
    print("   - pip install kaggle")
    print("   - Configura credenciales (kaggle.json)")
    print("   - kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
    print(f"   - unzip ... -d {KAGGLE_RAW_DIR}")
    print("   - python Multi-GNN/format_kaggle_files.py data/kaggle_raw/HI-Small_Trans.csv")
    
    print("\n⚠️ Por ahora, crearemos datos sintéticos para demostración...")
    USE_REAL_DATA = False

print("=" * 70)

## 2.2 Cargar o crear datos

Dependiendo de si tenemos datos reales de Kaggle o no, cargaremos los datos reales o crearemos datos sintéticos para demostración.

In [ ]:
def create_synthetic_data(n_transactions=5000, n_accounts=200, illicit_ratio=0.02):
    """
    Crea un dataset sintético de transacciones para demostración.
    
    Args:
        n_transactions: Número de transacciones
        n_accounts: Número de cuentas únicas
        illicit_ratio: Proporción de transacciones ilícitas
    
    Returns:
        DataFrame con transacciones sintéticas en formato compatible
    """
    np.random.seed(42)
    
    print("  Generando transacciones sintéticas...")
    
    # Generar timestamps ordenados (30 días)
    timestamps = np.sort(np.random.randint(0, 86400*30, n_transactions))
    
    # Generar IDs de cuentas
    from_ids = np.random.randint(0, n_accounts, n_transactions)
    to_ids = np.random.randint(0, n_accounts, n_transactions)
    
    # Asegurar que from_id != to_id
    same_mask = from_ids == to_ids
    to_ids[same_mask] = (to_ids[same_mask] + 1) % n_accounts
    
    # Generar monedas y formatos
    currencies = np.random.randint(0, 3, n_transactions)  # 3 monedas
    payment_formats = np.random.randint(0, 3, n_transactions)  # 3 formatos
    
    # Generar cantidades
    amounts = np.random.exponential(1000, n_transactions)
    
    # Generar etiquetas (desbalanceadas)
    n_illicit = int(n_transactions * illicit_ratio)
    labels = np.array([1] * n_illicit + [0] * (n_transactions - n_illicit))
    np.random.shuffle(labels)
    
    # Simular patrones: transacciones ilícitas tienden a ser más pequeñas
    illicit_mask = labels == 1
    amounts[illicit_mask] *= 0.3
    
    # Crear DataFrame en formato compatible con data_loading.py
    df = pd.DataFrame({
        'from_id': from_ids,
        'to_id': to_ids,
        'Timestamp': timestamps,
        'Amount Received': amounts,
        'Received Currency': currencies,
        'Payment Format': payment_formats,
        'Is Laundering': labels
    })
    
    return df

# Cargar o crear datos
if USE_REAL_DATA and FORMATTED_FILE.exists():
    print("\n📂 CARGANDO DATOS REALES DE KAGGLE")
    print("=" * 70)
    
    # Leer CSV formateado
    df_transactions = pd.read_csv(FORMATTED_FILE)
    
    print(f"✓ Datos cargados: {len(df_transactions):,} transacciones")
    print(f"  Columnas: {list(df_transactions.columns)}")
    
    # Verificar que tenga las columnas necesarias
    required_cols = ['from_id', 'to_id', 'Timestamp', 'Amount Received', 
                     'Received Currency', 'Payment Format', 'Is Laundering']
    
    missing_cols = [col for col in required_cols if col not in df_transactions.columns]
    if missing_cols:
        print(f"\n⚠️ Advertencia: Faltan columnas: {missing_cols}")
        print("El archivo puede necesitar formateo. Usa:")
        print(f"  python Multi-GNN/format_kaggle_files.py <archivo_crudo.csv>")
    
    # Mantener solo las columnas necesarias
    df_transactions = df_transactions[required_cols].copy()
    
else:
    print("\n🔨 CREANDO DATOS SINTÉTICOS")
    print("=" * 70)
    df_transactions = create_synthetic_data(n_transactions=5000, n_accounts=200)
    
    # Guardar para futuras ejecuciones
    synthetic_file = DATA_DIR / "synthetic_transactions.csv"
    df_transactions.to_csv(synthetic_file, index=False)
    print(f"✓ Datos sintéticos creados y guardados en: {synthetic_file}")

# Mostrar información básica
print(f"\n📊 RESUMEN DE DATOS")
print("=" * 70)
print(f"Total transacciones: {len(df_transactions):,}")
print(f"Transacciones ilícitas: {df_transactions['Is Laundering'].sum():,}")
print(f"Ratio ilícitas: {df_transactions['Is Laundering'].mean()*100:.3f}%")
print(f"\nRango temporal: {df_transactions['Timestamp'].min():.0f} - {df_transactions['Timestamp'].max():.0f}")
print(f"Duración: {(df_transactions['Timestamp'].max() - df_transactions['Timestamp'].min()) / 86400:.1f} días")
print(f"\nCuentas únicas:")
print(f"  - Orígenes: {df_transactions['from_id'].nunique():,}")
print(f"  - Destinos: {df_transactions['to_id'].nunique():,}")
print(f"  - Total: {len(set(df_transactions['from_id'].unique()) | set(df_transactions['to_id'].unique())):,}")

# Mostrar primeras filas
print("\n📋 Primeras 5 transacciones:")
print(df_transactions.head())

## 2.3 Análisis exploratorio

In [ ]:
# Estadísticas básicas
print("📈 ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 70)
print(f"\n1️⃣ DISTRIBUCIÓN DE CLASES:")
print(f"  Total transacciones: {len(df_transactions):,}")
print(f"  Transacciones ilícitas: {df_transactions['Is Laundering'].sum():,}")
print(f"  Transacciones lícitas: {(df_transactions['Is Laundering'] == 0).sum():,}")
print(f"  Ratio ilícitas: {df_transactions['Is Laundering'].mean()*100:.3f}%")

print(f"\n2️⃣ ANÁLISIS TEMPORAL:")
print(f"  Periodo: {df_transactions['Timestamp'].min():.0f} - {df_transactions['Timestamp'].max():.0f}")
print(f"  Duración: {(df_transactions['Timestamp'].max() - df_transactions['Timestamp'].min()) / 86400:.1f} días")

print(f"\n3️⃣ ANÁLISIS DE RED:")
n_from = df_transactions['from_id'].nunique()
n_to = df_transactions['to_id'].nunique()
n_total = len(set(df_transactions['from_id'].unique()) | set(df_transactions['to_id'].unique()))
print(f"  Cuentas origen: {n_from:,}")
print(f"  Cuentas destino: {n_to:,}")
print(f"  Cuentas únicas totales: {n_total:,}")
print(f"  Transacciones promedio por cuenta: {len(df_transactions) / n_total:.1f}")

print(f"\n4️⃣ ANÁLISIS DE CANTIDADES:")
print(f"  Media: ${df_transactions['Amount Received'].mean():,.2f}")
print(f"  Mediana: ${df_transactions['Amount Received'].median():,.2f}")
print(f"  Desv. estándar: ${df_transactions['Amount Received'].std():,.2f}")
print(f"  Min: ${df_transactions['Amount Received'].min():,.2f}")
print(f"  Max: ${df_transactions['Amount Received'].max():,.2f}")

# Comparar cantidades por clase
print(f"\n  Cantidades por clase:")
print(f"    Lícitas  - Media: ${df_transactions[df_transactions['Is Laundering']==0]['Amount Received'].mean():,.2f}")
print(f"    Ilícitas - Media: ${df_transactions[df_transactions['Is Laundering']==1]['Amount Received'].mean():,.2f}")

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Distribución de etiquetas
label_counts = df_transactions['Is Laundering'].value_counts()
axes[0,0].bar(['Lícita (0)', 'Ilícita (1)'], [label_counts.get(0, 0), label_counts.get(1, 0)], 
              color=['green', 'red'], alpha=0.7)
axes[0,0].set_title('Distribución de Clases', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('Número de transacciones')
axes[0,0].set_yscale('log')
for i, (label, count) in enumerate([(0, label_counts.get(0, 0)), (1, label_counts.get(1, 0))]):
    axes[0,0].text(i, count, f'{count:,}', ha='center', va='bottom')

# 2. Distribución de cantidades por clase
licit_amounts = df_transactions[df_transactions['Is Laundering']==0]['Amount Received']
illicit_amounts = df_transactions[df_transactions['Is Laundering']==1]['Amount Received']

axes[0,1].hist(licit_amounts, bins=50, alpha=0.6, label='Lícitas', color='green', density=True)
if len(illicit_amounts) > 0:
    axes[0,1].hist(illicit_amounts, bins=50, alpha=0.6, label='Ilícitas', color='red', density=True)
axes[0,1].set_title('Distribución de Cantidades por Clase', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Cantidad ($)')
axes[0,1].set_ylabel('Densidad')
axes[0,1].legend()
axes[0,1].set_xlim(0, df_transactions['Amount Received'].quantile(0.95))

# 3. Distribución de monedas
currency_counts = df_transactions['Received Currency'].value_counts().head(10)
axes[1,0].bar(range(len(currency_counts)), currency_counts.values, color='steelblue', alpha=0.7)
axes[1,0].set_title('Top 10 Monedas más usadas', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Moneda (ID)')
axes[1,0].set_ylabel('Número de transacciones')
axes[1,0].set_xticks(range(len(currency_counts)))
axes[1,0].set_xticklabels(currency_counts.index, rotation=45)

# 4. Distribución de formatos de pago
format_counts = df_transactions['Payment Format'].value_counts().head(10)
axes[1,1].bar(range(len(format_counts)), format_counts.values, color='coral', alpha=0.7)
axes[1,1].set_title('Top 10 Formatos de Pago', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Formato (ID)')
axes[1,1].set_ylabel('Número de transacciones')
axes[1,1].set_xticks(range(len(format_counts)))
axes[1,1].set_xticklabels(format_counts.index, rotation=45)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("⚠️ OBSERVACIONES IMPORTANTES:")
print("  1. Dataset muy DESBALANCEADO (~98% lícitas, ~2% ilícitas)")
print("  2. Usaremos pesos en la función de pérdida para compensar")
print("  3. F1-score será más relevante que accuracy")
print("  4. Las transacciones ilícitas tienden a tener cantidades menores")
print("=" * 70)

## 2.4 Convertir datos a formato de grafo

Ahora vamos a transformar las transacciones en un **grafo**:
- **Nodos**: Cuentas bancarias
- **Aristas**: Transacciones (dirigidas, de cuenta origen a cuenta destino)
- **Características de aristas**: Timestamp, Amount, Currency, Payment Format
- **Etiquetas**: Is Laundering (0 o 1)

In [ ]:
def transactions_to_graph(df):
    """
    Convierte DataFrame de transacciones a objeto PyG Data.
    Sigue exactamente el mismo proceso que data_loading.py de Multi-GNN.
    
    Args:
        df: DataFrame con columnas ['from_id', 'to_id', 'Timestamp', 
                                     'Amount Received', 'Received Currency', 
                                     'Payment Format', 'Is Laundering']
    
    Returns:
        data: Objeto torch_geometric.data.Data
    """
    print("🔧 Convirtiendo transacciones a grafo...")
    print("=" * 70)
    
    # 1. Normalizar timestamp (restar el mínimo, como en data_loading.py línea 21)
    df = df.copy()
    df['Timestamp'] = df['Timestamp'] - df['Timestamp'].min()
    
    # 2. Determinar número máximo de nodos
    max_n_id = df[['from_id', 'to_id']].max().max() + 1
    print(f"  Número de nodos (cuentas): {max_n_id:,}")
    
    # 3. Crear features de nodos (placeholder de 1s, como en data_loading.py línea 24)
    x = torch.ones(max_n_id, 1, dtype=torch.float)
    
    # 4. Crear edge_index
    edge_index = torch.tensor(df[['from_id', 'to_id']].values.T, dtype=torch.long)
    print(f"  Número de aristas (transacciones): {edge_index.shape[1]:,}")
    
    # 5. Etiquetas
    y = torch.tensor(df['Is Laundering'].values, dtype=torch.long)
    print(f"  Transacciones ilícitas: {y.sum().item():,} ({y.float().mean()*100:.3f}%)")
    
    # 6. Timestamps
    timestamps = torch.tensor(df['Timestamp'].values, dtype=torch.float)
    n_days = int(timestamps.max() / (3600 * 24) + 1)
    print(f"  Duración: {n_days} días")
    
    # 7. Crear edge attributes
    # Exactamente como en data_loading.py línea 32
    edge_features = []
    
    # Timestamp
    edge_features.append(df['Timestamp'].values.reshape(-1, 1))
    
    # Amount Received
    edge_features.append(df['Amount Received'].values.reshape(-1, 1))
    
    # Received Currency
    edge_features.append(df['Received Currency'].values.reshape(-1, 1))
    
    # Payment Format
    edge_features.append(df['Payment Format'].values.reshape(-1, 1))
    
    edge_attr = torch.tensor(np.concatenate(edge_features, axis=1), dtype=torch.float)
    print(f"  Dimensión de features de aristas: {edge_attr.shape[1]}")
    
    # 8. Crear objeto Data de PyTorch Geometric
    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
        num_nodes=max_n_id
    )
    
    # Guardar timestamps para división temporal
    data.timestamps = timestamps
    
    print("=" * 70)
    print("✓ Grafo creado exitosamente!")
    print(f"\n📊 Resumen del grafo:")
    print(f"  - Nodos (cuentas): {data.num_nodes:,}")
    print(f"  - Aristas (transacciones): {data.edge_index.shape[1]:,}")
    print(f"  - Features por nodo: {data.x.shape[1]}")
    print(f"  - Features por arista: {data.edge_attr.shape[1]}")
    print(f"  - Etiquetas positivas: {data.y.sum().item():,} ({data.y.float().mean()*100:.3f}%)")
    print(f"  - Grafo dirigido: Sí")
    print(f"  - Permite self-loops: No")
    
    return data

# Convertir datos a grafo
graph_data = transactions_to_graph(df_transactions)

# Verificar el objeto creado
print(f"\n🔍 Estructura del objeto Data:")
print(f"  {graph_data}")

# Verificar dimensiones
assert graph_data.x.shape[0] == graph_data.num_nodes, "Error: Dimensión de x no coincide con num_nodes"
assert graph_data.edge_index.shape[1] == graph_data.y.shape[0], "Error: Número de aristas no coincide con etiquetas"
assert graph_data.edge_attr.shape[0] == graph_data.y.shape[0], "Error: Features de aristas no coinciden con etiquetas"

print("\n✓ Todas las verificaciones pasaron correctamente")

## 2.5 División temporal de datos

En el código original, los datos se dividen **temporalmente** por días:
- **Train**: Primeros 60% de días
- **Validation**: Siguientes 20% de días
- **Test**: Últimos 20% de días

Esto simula un escenario realista: entrenar con datos históricos y predecir en datos futuros.

In [ ]:
def temporal_split(data, train_ratio=0.6, val_ratio=0.2):
    """
    Divide el grafo temporalmente basándose en timestamps.
    
    Args:
        data: Objeto PyG Data
        train_ratio: Proporción para entrenamiento
        val_ratio: Proporción para validación
    
    Returns:
        train_mask, val_mask, test_mask: Máscaras booleanas para cada split
    """
    timestamps = data.timestamps.numpy()
    sorted_indices = np.argsort(timestamps)
    
    n_edges = len(timestamps)
    n_train = int(n_edges * train_ratio)
    n_val = int(n_edges * val_ratio)
    
    train_mask = torch.zeros(n_edges, dtype=torch.bool)
    val_mask = torch.zeros(n_edges, dtype=torch.bool)
    test_mask = torch.zeros(n_edges, dtype=torch.bool)
    
    train_mask[sorted_indices[:n_train]] = True
    val_mask[sorted_indices[n_train:n_train+n_val]] = True
    test_mask[sorted_indices[n_train+n_val:]] = True
    
    return train_mask, val_mask, test_mask

# Crear máscaras de división
train_mask, val_mask, test_mask = temporal_split(graph_data)

graph_data.train_mask = train_mask
graph_data.val_mask = val_mask
graph_data.test_mask = test_mask

print("División temporal del dataset:")
print(f"  Train: {train_mask.sum()} transacciones ({train_mask.float().mean()*100:.1f}%)")
print(f"  Val:   {val_mask.sum()} transacciones ({val_mask.float().mean()*100:.1f}%)")
print(f"  Test:  {test_mask.sum()} transacciones ({test_mask.float().mean()*100:.1f}%)")

# Verificar distribución de clases en cada split
print("\nDistribución de clases ilícitas:")
print(f"  Train: {graph_data.y[train_mask].float().mean()*100:.2f}%")
print(f"  Val:   {graph_data.y[val_mask].float().mean()*100:.2f}%")
print(f"  Test:  {graph_data.y[test_mask].float().mean()*100:.2f}%")

---
# Parte 3: Comprensión de los Modelos

## 3.1 Arquitectura general

Todos los modelos Multi-GNN siguen esta estructura:

```
INPUT (Grafo)
    ↓
EMBEDDINGS (Transforman features a espacio latente)
    ↓
GNN LAYERS (Message Passing + Agregación)
    ├─ BatchNorm (normalización)
    ├─ ReLU (activación)
    └─ Residual Connection (skip connection)
    ↓
MLP FINAL (Clasificación de aristas)
    ├─ Linear(n_hidden*3, 50)
    ├─ BatchNorm + ReLU
    ├─ Linear(50, 25)
    ├─ BatchNorm + ReLU
    └─ Linear(25, 2)  → [P(lícita), P(ilícita)]
```

### ¿Por qué n_hidden*3 en el MLP?

Para clasificar una arista (i → j), concatenamos:
1. Embedding del nodo origen (i)
2. Embedding del nodo destino (j)
3. Features de la arista

Total: 3 × n_hidden dimensiones

## 3.2 Los 4 modelos explicados

### Modelo 1: GINe (Graph Isomorphism Network with Edge features)

**Idea principal**: Basado en el algoritmo Weisfeiler-Lehman para testing de isomorfismo de grafos.

**Fórmula de actualización**:
```
h_i^(k+1) = MLP( (1 + ε) · h_i^(k) + Σ_{j∈N(i)} MLP(h_j^(k) + e_ij) )
```

Donde:
- `h_i`: Embedding del nodo i
- `e_ij`: Features de la arista i→j
- `ε`: Parámetro aprendible
- `MLP`: Red neuronal pequeña

**Ventajas**:
- Teóricamente muy expresivo
- Rápido y eficiente
- Buen baseline

**Parámetros clave**:
- `n_hidden=66`: Dimensión de embeddings
- `n_gnn_layers=2`: Número de capas GNN

### Modelo 2: GATe (Graph Attention Network with Edge features)

**Idea principal**: Usa **atención multi-cabeza** para ponderar la importancia de cada vecino.

**Fórmula de atención**:
```
α_ij = softmax_j( LeakyReLU( a^T [Wh_i || Wh_j || e_ij] ) )
h_i^(k+1) = Σ_{j∈N(i)} α_ij · W · h_j^(k)
```

Donde:
- `α_ij`: Peso de atención (¿cuán importante es el vecino j para i?)
- `||`: Concatenación
- `W`: Matriz de pesos aprendible

**Ventajas**:
- Aprende qué vecinos son más relevantes
- Multi-cabeza captura diferentes aspectos
- Interpretable (podemos visualizar atención)

**Parámetros clave**:
- `n_heads=4`: Número de cabezas de atención
- `n_hidden=64`: Dimensión por cabeza

### Modelo 3: PNA (Principal Neighbourhood Aggregation)

**Idea principal**: En lugar de una sola función de agregación, usa **múltiples agregadores**.

**Agregadores**:
- `mean`: Promedio de vecinos
- `max`: Máximo valor
- `min`: Mínimo valor
- `std`: Desviación estándar

**Escaladores** (para normalizar por grado del nodo):
- `identity`: Sin escalar
- `amplification`: Amplifica nodos con muchos vecinos
- `attenuation`: Atenúa nodos con muchos vecinos

**Ventajas**:
- Más expresivo que GIN o GAT
- Captura diferentes aspectos del vecindario
- Estado del arte en varios benchmarks

**Parámetros clave**:
- `n_hidden=20`: Dimensión (menor porque concatena múltiples agregadores)

### Modelo 4: RGCN (Relational Graph Convolutional Network)

**Idea principal**: Diseñado para **grafos multi-relacionales** (con diferentes tipos de aristas).

**Fórmula**:
```
h_i^(k+1) = σ( Σ_{r∈R} Σ_{j∈N_r(i)} W_r · h_j^(k) / |N_r(i)| )
```

Donde:
- `r`: Tipo de relación
- `W_r`: Pesos específicos para la relación r
- `N_r(i)`: Vecinos de i conectados por relación r

**En Multi-GNN**: Se usa con `--reverse_mp` para crear 2 tipos de aristas:
1. `node → node`: Transacción normal
2. `node ←rev→ node`: Transacción inversa (para capturar flujo de dinero en ambas direcciones)

**Ventajas**:
- Maneja múltiples tipos de relaciones
- Útil para grafos heterogéneos

**Parámetros clave**:
- `num_relations`: Número de tipos de aristas
- `n_hidden=66`: Dimensión

## 3.3 Implementación simplificada de GINe

Vamos a implementar una versión simplificada de GINe para entender cómo funciona.

In [ ]:
from torch_geometric.nn import GINEConv, global_add_pool

class SimpleGINe(nn.Module):
    """
    Versión simplificada de GINe para propósitos educativos.
    """
    def __init__(self, n_features, n_hidden, n_edge_features, n_layers=2):
        super().__init__()
        
        print("\n🏗️ Construyendo modelo GINe...")
        print(f"  - Input features: {n_features}")
        print(f"  - Hidden dimension: {n_hidden}")
        print(f"  - Edge features: {n_edge_features}")
        print(f"  - GNN layers: {n_layers}")
        
        # Embedding inicial de nodos
        self.node_emb = nn.Linear(n_features, n_hidden)
        
        # Embedding de características de aristas
        self.edge_emb = nn.Linear(n_edge_features, n_hidden)
        
        # Capas GNN
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        for i in range(n_layers):
            # MLP para GINE
            mlp = nn.Sequential(
                nn.Linear(n_hidden, n_hidden),
                nn.BatchNorm1d(n_hidden),
                nn.ReLU(),
                nn.Linear(n_hidden, n_hidden)
            )
            self.convs.append(GINEConv(mlp, train_eps=True))
            self.batch_norms.append(nn.BatchNorm1d(n_hidden))
        
        # MLP final para clasificación de aristas
        self.edge_classifier = nn.Sequential(
            nn.Linear(n_hidden * 3, 50),  # 3x porque concatenamos: src, dst, edge
            nn.BatchNorm1d(50),
            nn.ReLU(),
            nn.Linear(50, 25),
            nn.BatchNorm1d(25),
            nn.ReLU(),
            nn.Linear(25, 2)  # 2 clases: lícita vs ilícita
        )
        
        print("✓ Modelo construido")
    
    def forward(self, x, edge_index, edge_attr):
        """
        Forward pass del modelo.
        
        Args:
            x: Features de nodos [num_nodes, n_features]
            edge_index: Índices de aristas [2, num_edges]
            edge_attr: Features de aristas [num_edges, n_edge_features]
        
        Returns:
            logits: Predicciones para cada arista [num_edges, 2]
        """
        # 1. Embeddings iniciales
        x = self.node_emb(x)  # [num_nodes, n_hidden]
        edge_emb = self.edge_emb(edge_attr)  # [num_edges, n_hidden]
        
        # 2. Message passing a través de capas GNN
        for i, conv in enumerate(self.convs):
            x_prev = x  # Guardar para residual connection
            
            # Convolución GNN
            x = conv(x, edge_index, edge_emb)
            
            # Normalización y activación
            x = self.batch_norms[i](x)
            x = F.relu(x)
            
            # Residual connection (si las dimensiones coinciden)
            if x.shape == x_prev.shape:
                x = x + x_prev
        
        # 3. Clasificación de aristas
        # Para cada arista (i → j), concatenamos: h_i, h_j, edge_emb
        src, dst = edge_index[0], edge_index[1]
        edge_repr = torch.cat([
            x[src],      # Embedding del nodo origen
            x[dst],      # Embedding del nodo destino
            edge_emb     # Embedding de la arista
        ], dim=1)  # [num_edges, n_hidden * 3]
        
        # 4. Predicción final
        logits = self.edge_classifier(edge_repr)  # [num_edges, 2]
        
        return logits

# Crear instancia del modelo
model = SimpleGINe(
    n_features=graph_data.x.shape[1],
    n_hidden=64,
    n_edge_features=graph_data.edge_attr.shape[1],
    n_layers=2
)

# Contar parámetros
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Estadísticas del modelo:")
print(f"  - Total de parámetros: {n_params:,}")
print(f"  - Parámetros entrenables: {n_trainable:,}")

# Test forward pass
print("\n🧪 Probando forward pass...")
with torch.no_grad():
    output = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    print(f"  Input: {graph_data.num_nodes} nodos, {graph_data.edge_index.shape[1]} aristas")
    print(f"  Output shape: {output.shape} (una predicción por arista)")
    print(f"  Output example (primeras 3 aristas):")
    print(f"    {output[:3]}")
    print(f"\n  Interpretación: [logit_clase_0, logit_clase_1]")
    print(f"  Clase predicha (primeras 3): {output[:3].argmax(dim=1)}")

---
# Parte 4: Entrenamiento Completo

## 4.1 Configuración del entrenamiento

In [ ]:
# Configuración
config = {
    'lr': 0.006,              # Learning rate
    'n_epochs': 50,           # Número de épocas
    'batch_size': 128,        # Tamaño de batch (para LinkNeighborLoader)
    'n_hidden': 64,           # Dimensión oculta
    'n_gnn_layers': 2,        # Capas GNN
    'num_neighbors': [50, 50], # Vecinos a samplear por capa
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Pesos para Cross Entropy (por desbalance de clases)
    'w_ce1': 1.0,             # Peso para clase 0 (lícita)
    'w_ce2': 6.0,             # Peso para clase 1 (ilícita) - mayor para compensar
}

print("⚙️ CONFIGURACIÓN DE ENTRENAMIENTO")
print("=" * 50)
for key, value in config.items():
    print(f"  {key:20s}: {value}")

# Mover datos a device
device = torch.device(config['device'])
graph_data = graph_data.to(device)
model = model.to(device)

print(f"\n✓ Datos y modelo movidos a: {device}")

## 4.2 Función de pérdida y optimizador

In [ ]:
# Pesos para compensar desbalance de clases
class_weights = torch.tensor([config['w_ce1'], config['w_ce2']]).to(device)

# Cross Entropy Loss con pesos
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizador Adam
optimizer = Adam(model.parameters(), lr=config['lr'])

print("✓ Loss function: CrossEntropyLoss con pesos")
print(f"  - Peso clase 0 (lícita): {config['w_ce1']}")
print(f"  - Peso clase 1 (ilícita): {config['w_ce2']}")
print(f"\n✓ Optimizador: Adam con lr={config['lr']}")

## 4.3 Funciones de evaluación

In [ ]:
def evaluate(model, data, mask):
    """
    Evalúa el modelo en un subset de datos.
    
    Args:
        model: Modelo a evaluar
        data: Objeto Data con el grafo completo
        mask: Máscara booleana para seleccionar aristas
    
    Returns:
        metrics: Dict con métricas (loss, f1, precision, recall)
    """
    model.eval()
    
    with torch.no_grad():
        # Forward pass
        logits = model(data.x, data.edge_index, data.edge_attr)
        
        # Filtrar por máscara
        logits_masked = logits[mask]
        labels_masked = data.y[mask]
        
        # Loss
        loss = criterion(logits_masked, labels_masked)
        
        # Predicciones
        preds = logits_masked.argmax(dim=1)
        
        # Métricas (convertir a CPU y numpy para sklearn)
        preds_cpu = preds.cpu().numpy()
        labels_cpu = labels_masked.cpu().numpy()
        
        f1 = f1_score(labels_cpu, preds_cpu, average='binary', zero_division=0)
        precision = precision_score(labels_cpu, preds_cpu, average='binary', zero_division=0)
        recall = recall_score(labels_cpu, preds_cpu, average='binary', zero_division=0)
        
        # Accuracy
        accuracy = (preds == labels_masked).float().mean().item()
    
    return {
        'loss': loss.item(),
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'accuracy': accuracy
    }

print("✓ Función de evaluación definida")
print("  Métricas: Loss, F1-score, Precision, Recall, Accuracy")

## 4.4 Loop de entrenamiento

In [ ]:
def train_epoch(model, data, train_mask, optimizer):
    """
    Entrena el modelo por una época.
    """
    model.train()
    
    # Forward pass
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index, data.edge_attr)
    
    # Calcular loss solo en training set
    loss = criterion(logits[train_mask], data.y[train_mask])
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Calcular F1 en train
    with torch.no_grad():
        preds = logits[train_mask].argmax(dim=1)
        labels = data.y[train_mask]
        f1 = f1_score(labels.cpu().numpy(), preds.cpu().numpy(), average='binary', zero_division=0)
    
    return loss.item(), f1

# Historial de entrenamiento
history = {
    'train_loss': [],
    'train_f1': [],
    'val_loss': [],
    'val_f1': [],
    'val_precision': [],
    'val_recall': [],
}

best_val_f1 = 0.0
best_epoch = 0

print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 70)

for epoch in range(config['n_epochs']):
    # Entrenar
    train_loss, train_f1 = train_epoch(model, graph_data, train_mask, optimizer)
    
    # Evaluar en validación
    val_metrics = evaluate(model, graph_data, val_mask)
    
    # Guardar historial
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_metrics['loss'])
    history['val_f1'].append(val_metrics['f1'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    
    # Guardar mejor modelo
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_epoch = epoch
        # En producción, aquí guardarías el modelo: torch.save(model.state_dict(), 'best_model.pt')
    
    # Imprimir progreso cada 5 épocas
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{config['n_epochs']} | "
              f"Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | "
              f"Val Loss: {val_metrics['loss']:.4f} F1: {val_metrics['f1']:.4f} "
              f"P: {val_metrics['precision']:.4f} R: {val_metrics['recall']:.4f}")

print("\n✓ ENTRENAMIENTO COMPLETADO")
print(f"  Mejor época: {best_epoch+1} con Val F1: {best_val_f1:.4f}")

## 4.5 Visualización del entrenamiento

In [ ]:
# Visualizar curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch+1})')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curva de Pérdida')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 Score
axes[1].plot(history['train_f1'], label='Train F1', linewidth=2)
axes[1].plot(history['val_f1'], label='Val F1', linewidth=2)
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch+1})')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score durante Entrenamiento')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Visualizar Precision vs Recall
plt.figure(figsize=(10, 5))
plt.plot(history['val_precision'], label='Precision', linewidth=2)
plt.plot(history['val_recall'], label='Recall', linewidth=2)
plt.axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch ({best_epoch+1})')
plt.xlabel('Época')
plt.ylabel('Score')
plt.title('Precision y Recall en Validación')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n📊 Interpretación:")
print("  - Loss: Debe decrecer. Si aumenta en val, hay overfitting.")
print("  - F1: Balance entre precision y recall. Mejor métrica para datos desbalanceados.")
print("  - Precision: De las predichas ilícitas, ¿cuántas son realmente ilícitas?")
print("  - Recall: De las realmente ilícitas, ¿cuántas detectamos?")

---
# Parte 5: Evaluación Final y Análisis

## 5.1 Evaluación en Test Set

In [ ]:
# Evaluar en test set
test_metrics = evaluate(model, graph_data, test_mask)

print("🎯 RESULTADOS FINALES EN TEST SET")
print("=" * 50)
print(f"  Loss:      {test_metrics['loss']:.4f}")
print(f"  F1 Score:  {test_metrics['f1']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")

# Matriz de confusión
model.eval()
with torch.no_grad():
    logits = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    preds = logits[test_mask].argmax(dim=1).cpu().numpy()
    labels = graph_data.y[test_mask].cpu().numpy()

cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Lícita', 'Ilícita'],
            yticklabels=['Lícita', 'Ilícita'])
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción')
plt.title('Matriz de Confusión - Test Set')
plt.show()

print("\n📊 Interpretación de la Matriz de Confusión:")
print(f"  True Negatives (TN):  {cm[0,0]:5d} - Lícitas correctamente clasificadas")
print(f"  False Positives (FP): {cm[0,1]:5d} - Lícitas clasificadas como ilícitas (falsa alarma)")
print(f"  False Negatives (FN): {cm[1,0]:5d} - Ilícitas clasificadas como lícitas (¡peligroso!)")
print(f"  True Positives (TP):  {cm[1,1]:5d} - Ilícitas correctamente detectadas")

## 5.2 Análisis de predicciones

In [ ]:
# Obtener probabilidades
model.eval()
with torch.no_grad():
    logits = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    probs = F.softmax(logits, dim=1)
    
    # Probabilidad de ser ilícita
    prob_illicit = probs[:, 1].cpu().numpy()

# Visualizar distribución de probabilidades
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribución por clase real
test_prob_licit = prob_illicit[test_mask.cpu().numpy() & (graph_data.y.cpu().numpy() == 0)]
test_prob_illicit = prob_illicit[test_mask.cpu().numpy() & (graph_data.y.cpu().numpy() == 1)]

axes[0].hist(test_prob_licit, bins=50, alpha=0.6, label='Lícitas (real)', color='green')
axes[0].hist(test_prob_illicit, bins=50, alpha=0.6, label='Ilícitas (real)', color='red')
axes[0].axvline(0.5, color='black', linestyle='--', label='Umbral (0.5)')
axes[0].set_xlabel('Probabilidad de ser Ilícita')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Probabilidades - Test Set')
axes[0].legend()
axes[0].set_yscale('log')

# Top transacciones más sospechosas
test_indices = torch.where(test_mask)[0].cpu().numpy()
top_k = 20
top_suspicious = np.argsort(prob_illicit[test_mask.cpu().numpy()])[-top_k:][::-1]
top_suspicious_global = test_indices[top_suspicious]

top_probs = prob_illicit[test_indices[top_suspicious]]
top_labels = graph_data.y[test_mask][top_suspicious].cpu().numpy()

colors = ['red' if label == 1 else 'orange' for label in top_labels]
axes[1].barh(range(top_k), top_probs, color=colors)
axes[1].set_xlabel('Probabilidad de ser Ilícita')
axes[1].set_ylabel('Ranking')
axes[1].set_title(f'Top {top_k} Transacciones Más Sospechosas')
axes[1].axvline(0.5, color='black', linestyle='--', alpha=0.5)
axes[1].invert_yaxis()

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Realmente Ilícita'),
    Patch(facecolor='orange', label='Realmente Lícita (Falso Positivo)')
]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

print(f"\n🔍 Top {top_k} transacciones más sospechosas:")
print(f"  Correctas (TP): {(top_labels == 1).sum()} / {top_k}")
print(f"  Falsas alarmas (FP): {(top_labels == 0).sum()} / {top_k}")

## 5.3 Inferencia en nuevas transacciones

Así es como usarías el modelo entrenado para clasificar nuevas transacciones.

In [ ]:
def predict_transaction(model, graph_data, edge_index_to_predict):
    """
    Predice si una transacción específica es ilícita.
    
    Args:
        model: Modelo entrenado
        graph_data: Grafo completo
        edge_index_to_predict: Índice de la arista a predecir
    
    Returns:
        prediction: 0 (lícita) o 1 (ilícita)
        probability: Probabilidad de ser ilícita
    """
    model.eval()
    
    with torch.no_grad():
        # Forward pass
        logits = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
        
        # Obtener predicción para la arista específica
        logit = logits[edge_index_to_predict]
        prob = F.softmax(logit, dim=0)
        
        prediction = logit.argmax().item()
        probability = prob[1].item()  # Probabilidad de clase 1 (ilícita)
    
    return prediction, probability

# Ejemplo: predecir algunas transacciones del test set
print("🔮 EJEMPLOS DE PREDICCIÓN")
print("=" * 70)

# Seleccionar 5 transacciones aleatorias del test set
test_indices = torch.where(test_mask)[0]
sample_indices = test_indices[torch.randperm(len(test_indices))[:5]]

for i, idx in enumerate(sample_indices):
    # Información de la transacción
    src, dst = graph_data.edge_index[:, idx]
    real_label = graph_data.y[idx].item()
    
    # Predicción
    pred_label, pred_prob = predict_transaction(model, graph_data, idx)
    
    # Mostrar
    print(f"\nTransacción {i+1}:")
    print(f"  Cuenta origen → destino: {src.item()} → {dst.item()}")
    print(f"  Etiqueta real: {'Ilícita' if real_label == 1 else 'Lícita'}")
    print(f"  Predicción: {'Ilícita' if pred_label == 1 else 'Lícita'} (prob={pred_prob:.3f})")
    print(f"  ¿Correcto? {'✓' if pred_label == real_label else '✗'}")

print("\n" + "=" * 70)

---
# Parte 6: Usando el Código Original Multi-GNN

## 6.1 Cómo ejecutar el código original

Ahora que entiendes cómo funcionan los modelos, puedes ejecutar el código original de Multi-GNN.

### Preparación de datos:

1. Descarga los datos de Kaggle
2. Formatea el CSV (si es necesario)
3. Actualiza `Multi-GNN/data_config.json` con las rutas correctas

### Comandos de entrenamiento:

```bash
# Ir al directorio Multi-GNN
cd Multi-GNN

# Entrenar modelo GIN básico
python main.py --data Small_HI --model gin

# Entrenar GIN con todas las adaptaciones Multi-GNN
python main.py --data Small_HI --model gin --emlps --reverse_mp --ego --ports --tds

# Entrenar otros modelos
python main.py --data Small_HI --model gat
python main.py --data Small_HI --model pna
python main.py --data Small_HI --model rgcn --reverse_mp

# Guardar modelo
python main.py --data Small_HI --model gin --save_model --unique_name my_gin_model

# Inferencia con modelo guardado
python main.py --data Small_HI --model gin --inference --unique_name my_gin_model
```

### Parámetros importantes:

- `--data`: Nombre del dataset (carpeta en data_config.json)
- `--model`: gin, gat, pna, o rgcn
- `--emlps`: Activa Edge MLPs
- `--reverse_mp`: Message passing inverso (grafo heterogéneo)
- `--ego`: Añade ego IDs a nodos centrales
- `--ports`: Numeración de puertos (orden temporal de transacciones)
- `--tds`: Time deltas entre transacciones
- `--n_epochs`: Número de épocas (default: 100)
- `--batch_size`: Tamaño de batch (default: 8192)

## 6.2 Estructura de archivos esperada

```
Multi-GNN/
├── main.py              # Punto de entrada
├── models.py            # Definiciones de modelos
├── training.py          # Lógica de entrenamiento
├── data_loading.py      # Carga de datos
├── data_util.py         # Utilidades de datos
├── train_util.py        # Utilidades de entrenamiento
├── inference.py         # Inferencia
├── util.py              # Utilidades generales
├── data_config.json     # Configuración de rutas de datos
├── model_settings.json  # Hiperparámetros por modelo
└── env.yml              # Dependencias conda

data/
└── formatted_transactions.csv  # Tus datos

logs/
└── logs.log             # Logs de entrenamiento

chkpts/
└── [modelos guardados]  # Checkpoints
```

## 6.3 Personalizar hiperparámetros

Edita `Multi-GNN/model_settings.json`:

```json
{
  "gin": {
    "params": {
      "lr": 0.00621,        // Learning rate
      "n_hidden": 66,       // Dimensión oculta
      "n_gnn_layers": 2,    // Capas GNN
      "w_ce1": 1.0,         // Peso clase lícita
      "w_ce2": 6.27         // Peso clase ilícita
    }
  }
}
```

---
# Conclusiones y Próximos Pasos

## ¿Qué aprendiste?

1. **Conceptos de GNN**: Message passing, agregación, edge prediction
2. **Datos como grafos**: Transacciones → aristas, cuentas → nodos
3. **4 arquitecturas**: GINe, GATe, PNA, RGCN
4. **Entrenamiento**: Loss, optimización, evaluación
5. **Métricas**: F1, precision, recall (más importantes que accuracy)
6. **Desbalance de clases**: Uso de pesos en loss function

## Próximos pasos sugeridos:

### 1. Experimentar con datos reales
- Descarga el dataset de Kaggle
- Ejecuta el código original con diferentes modelos
- Compara resultados

### 2. Probar adaptaciones Multi-GNN
- `--emlps`: Mejora actualización de aristas
- `--ports`: Captura orden temporal
- `--tds`: Captura tiempo entre transacciones
- `--reverse_mp`: Message passing bidireccional

### 3. Optimización de hiperparámetros
- Ajusta learning rate
- Prueba diferentes dimensiones ocultas
- Varía el número de capas GNN
- Experimenta con pesos de clase

### 4. Análisis avanzado
- Visualiza embeddings con t-SNE/UMAP
- Analiza errores (FP y FN)
- Estudia patrones de transacciones ilícitas detectadas
- Interpreta pesos de atención (en GAT)

### 5. Extensiones
- Añade más features (geolocalización, tipo de negocio, etc.)
- Implementa técnicas de explicabilidad (GNNExplainer)
- Prueba otras arquitecturas (GraphSAINT, GraphSAGE)
- Deploy en producción con API REST

## Recursos adicionales:

- **Paper Multi-GNN**: [Buscar en Google Scholar]
- **PyTorch Geometric**: https://pytorch-geometric.readthedocs.io/
- **GNN explicado**: http://snap.stanford.edu/class/cs224w-2021/
- **Kaggle dataset**: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml

## Preguntas frecuentes:

**P: ¿Por qué F1 y no accuracy?**
R: Porque los datos están muy desbalanceados. Un modelo que prediga todo como "lícita" tendría 99%+ accuracy pero sería inútil. F1 balancea precision y recall.

**P: ¿Cuántas épocas debo entrenar?**
R: Depende. Usa early stopping: para cuando val_loss deje de mejorar por N épocas (e.g., 10).

**P: ¿GPU es necesaria?**
R: No es estrictamente necesaria para datasets pequeños, pero acelera mucho el entrenamiento (10-100x).

**P: ¿Qué modelo usar?**
R: Empieza con GIN (más simple). Si no funciona bien, prueba GAT o PNA. RGCN solo si tienes múltiples tipos de relaciones.

**P: ¿Cómo interpretar las predicciones?**
R: La probabilidad indica confianza. Un threshold de 0.5 es estándar, pero puedes ajustarlo según el trade-off precision/recall que necesites.

---

## ¡Éxito con tus experimentos! 🚀

Si tienes dudas, consulta la documentación de PyTorch Geometric o el código original en `Multi-GNN/`.